# Role-Tagged Latent Injection

**A small transformer reaches chain-of-thought accuracy on 4-step composition while emitting zero extra tokens — and the reason is not what the classical theory predicts.**

---

## The result (reproduced on GPU, 3 seeds)

| arm | S5 (chance 0.20) | Affine (chance 0.059) |
|---|---|---|
| baseline | 0.2025 | 0.0620 |
| 1 slot, plain sum | 0.6305 | 0.0910 |
| 3 separate slots, raw values | 0.5180 | 0.1180 |
| **role-tagged latents** | **1.0000** | **1.0000** |
| token chain-of-thought | 1.0000 | 1.0000 |

Seeds: 1.0000 / 1.0000 / 1.0000 (S5) and 1.0000 / 0.9975 / 0.9985 (Affine).

## What the mechanism actually is

The classical variable-binding story — bind, superpose into one vector, unbind by
role — **is wrong here**, and the controls in section 8 kill it:

| control | result | what it rules out |
|---|---|---|
| unbinding roles shuffled | 1.0000 | retrieval-by-role is not used |
| no superposition at all | 1.0000 | superposition is not needed |
| bind only, never unbound | 1.0000 | unbinding is not needed |

What survives is narrower and cleaner:

| control | result |
|---|---|
| distinct random **orthogonal** matrix per slot | **1.0000** |
| distinct random **dimension permutation** per slot | **1.0000** |
| distinct **circular convolution** per slot | **1.0000** |
| **same** transform on every slot | **0.4815** |
| raw values, no transform | 0.4745 |
| raw values scaled x0.25 (norm-matched) | 0.4400 |

> **Each injected value must carry a role-distinct signature. Any distinct
> invertible transform works. The same transform on every slot fails.
> Positional separation alone is not enough — the value itself must be tagged.**

Scale is ruled out: the winning arm and the failing raw arm have almost identical
slot norms (8.84 vs 8.22), and shrinking the raw values made things worse.

## What this does NOT show

- **The intermediates are handed to the model, not computed by it.** This measures
  whether a model can *use* a value it has — the original puzzle, since a model
  with 100% auxiliary accuracy still failed. It does not show the model can produce
  them. That is the open problem.
- **Toy scale.** 64-dim, 2 layers, 5-state and 17-state synthetic algebra.
- **Looping still fails here.** Section 9 tested recurrent depth properly
  (zero-init, 12,000 steps) and it stayed at chance — 0.2055. The grokking result
  reported elsewhere did not reproduce at this scale.

## 1 · Setup

In [ ]:
import math, time, json, itertools, types
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337); np.random.seed(1337)
print("device:", DEV)

## 2 · HRR ops (one of several transforms tested — section 8 shows it is not special)

In [ ]:
def hrr_bind(x, y):
    n = x.shape[-1]
    return torch.fft.irfft(torch.fft.rfft(x, n=n) * torch.fft.rfft(y, n=n), n=n)
def hrr_inv(y):
    return torch.cat([y[..., :1], y[..., 1:].flip(-1)], dim=-1)
def hrr_unbind(b, y):
    return hrr_bind(b, hrr_inv(y))
print("[ok]")

## 3 · Two task families (different algebra — a result on both is not an artifact of one)

In [ ]:
_P=list(itertools.permutations(range(5))); PERM=np.array(_P,dtype=np.int64); N_POS=5

class S5Task:
    name="S5"
    def __init__(self,k=4,domain=120,seed=0):
        self.k,self.domain,self.n_states=k,domain,N_POS
        self.POFF,self.BOS,self.SEP,self.VOCAB=5,125,126,127
        self.rng=np.random.default_rng(seed)
    def sample(self,n): return (self.rng.integers(0,N_POS,size=n),
                                self.rng.integers(0,self.domain,size=(n,self.k)))
    def trace(self,s,g):
        out=np.empty_like(g); cur=s.copy()
        for t in range(g.shape[1]): cur=PERM[g[:,t],cur]; out[:,t]=cur
        return out
    def batch(self,n,dev):
        s,g=self.sample(n); tr=self.trace(s,g)
        seq=np.concatenate([np.full((n,1),self.BOS),s[:,None],g+self.POFF,np.full((n,1),self.SEP)],1)
        return (torch.from_numpy(seq).long().to(dev),torch.from_numpy(tr[:,-1]).long().to(dev),
                torch.from_numpy(tr[:,:-1]).long().to(dev))
    def batch_cot(self,n,dev):
        s,g=self.sample(n); tr=self.trace(s,g)
        seq=np.concatenate([np.full((n,1),self.BOS),s[:,None],g+self.POFF,np.full((n,1),self.SEP),tr],1)
        return (torch.from_numpy(seq[:,:-1]).long().to(dev),torch.from_numpy(seq[:,1:]).long().to(dev))
    @property
    def maxlen(self): return 2*self.k+7
    @property
    def chance(self): return 1.0/N_POS

class AffineTask:
    name="Affine"
    def __init__(self,k=4,m=17,domain=120,seed=0):
        self.k,self.m,self.domain,self.n_states=k,m,domain,m
        self.rng=np.random.default_rng(seed); pairs,a=[],1
        while len(pairs)<domain:
            for b in range(m):
                if len(pairs)>=domain: break
                if a%m!=0: pairs.append((a%m,b))
            a+=1
        self.pool=np.array(pairs[:domain],dtype=np.int64)
        self.BOS,self.SEP=m,m+1; self.POFF=m+2; self.VOCAB=self.POFF+domain
    def sample(self,n): return (self.rng.integers(0,self.m,size=n),
                                self.rng.integers(0,self.domain,size=(n,self.k)))
    def trace(self,x0,ops):
        out=np.empty_like(ops); cur=x0.copy()
        for t in range(ops.shape[1]):
            a=self.pool[ops[:,t],0]; b=self.pool[ops[:,t],1]
            cur=(a*cur+b)%self.m; out[:,t]=cur
        return out
    def batch(self,n,dev):
        x0,ops=self.sample(n); tr=self.trace(x0,ops)
        seq=np.concatenate([np.full((n,1),self.BOS),x0[:,None],ops+self.POFF,np.full((n,1),self.SEP)],1)
        return (torch.from_numpy(seq).long().to(dev),torch.from_numpy(tr[:,-1]).long().to(dev),
                torch.from_numpy(tr[:,:-1]).long().to(dev))
    def batch_cot(self,n,dev):
        x0,ops=self.sample(n); tr=self.trace(x0,ops)
        seq=np.concatenate([np.full((n,1),self.BOS),x0[:,None],ops+self.POFF,np.full((n,1),self.SEP),tr],1)
        return (torch.from_numpy(seq[:,:-1]).long().to(dev),torch.from_numpy(seq[:,1:]).long().to(dev))
    @property
    def maxlen(self): return 2*self.k+7
    @property
    def chance(self): return 1.0/self.m
print("[ok]")

## 4 · Model — everything hinges on `_memory_slots`

In [ ]:
class Block(nn.Module):
    def __init__(self,d,h,zero_init=False):
        super().__init__()
        self.n1=nn.LayerNorm(d); self.attn=nn.MultiheadAttention(d,h,batch_first=True)
        self.n2=nn.LayerNorm(d); self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d))
        if zero_init:
            nn.init.zeros_(self.attn.out_proj.weight); nn.init.zeros_(self.attn.out_proj.bias)
            nn.init.zeros_(self.ff[-1].weight); nn.init.zeros_(self.ff[-1].bias)
    def forward(self,x,mask):
        h=self.n1(x); a,_=self.attn(h,h,h,attn_mask=mask,need_weights=False)
        x=x+a; return x+self.ff(self.n2(x))

class BindingLM(nn.Module):
    def __init__(self,task,d=64,heads=4,mode="baseline",n_loops=1,zero_init=False,seed=0):
        super().__init__()
        self.task,self.d,self.mode=task,d,mode
        self.n_inter=task.k-1; self.n_loops=n_loops
        self.tok=nn.Embedding(task.VOCAB,d); self.pos=nn.Embedding(task.maxlen+8,d)
        self.l1=Block(d,heads,zero_init and n_loops>1); self.l2=Block(d,heads,zero_init and n_loops>1)
        self.norm=nn.LayerNorm(d); self.head=nn.Linear(d,task.VOCAB)
        g=torch.Generator().manual_seed(seed+999)
        self.register_buffer("roles",torch.randn(max(self.n_inter,1),d,generator=g)/math.sqrt(d))
        self.val_embed=nn.Embedding(task.n_states,d)
    def _memory_slots(self,f):
        if self.mode=="raw_multi":   return torch.stack(f,1)
        if self.mode=="sum_noroles": return torch.stack(f,1).sum(1,keepdim=True)
        if self.mode=="hrr":
            M=sum(hrr_bind(x,self.roles[i]) for i,x in enumerate(f))
            return torch.stack([hrr_unbind(M,self.roles[i]) for i in range(len(f))],1)
        return None
    def forward(self,idx,oracle_inter=None):
        B,T=idx.shape
        x=self.tok(idx)+self.pos(torch.arange(T,device=idx.device))[None]
        m=torch.triu(torch.full((T,T),float("-inf"),device=idx.device),1)
        for _ in range(self.n_loops): x=self.l1(x,m)
        if self.mode not in ("baseline","cot") and oracle_inter is not None:
            f=[self.val_embed(oracle_inter[:,i]) for i in range(self.n_inter)]
            s=self._memory_slots(f)
            if s is not None:
                x=torch.cat([s,x],1); T2=x.shape[1]
                m=torch.triu(torch.full((T2,T2),float("-inf"),device=idx.device),1)
        for _ in range(self.n_loops): x=self.l2(x,m)
        return self.head(self.norm(x)),None
    def n_params(self): return sum(p.numel() for p in self.parameters())
print("[ok]")

## 5 · Train / evaluate

In [ ]:
def fit(task,mode,steps=2500,d=64,bs=256,lr=3e-3,n_loops=1,zero_init=False,seed=0,
        slot_fn=None,dev=DEV):
    torch.manual_seed(seed); np.random.seed(seed)
    m=BindingLM(task,d=d,mode=mode,n_loops=n_loops,zero_init=zero_init,seed=seed).to(dev)
    if slot_fn is not None: m._memory_slots=types.MethodType(slot_fn,m)
    opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=0.01)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,lr,total_steps=steps)
    ns=task.n_states
    for s in range(steps):
        if mode=="cot":
            x,y=task.batch_cot(bs,dev); lg,_=m(x)
            mk=torch.zeros_like(y,dtype=torch.bool); mk[:,-task.k:]=True
            loss=F.cross_entropy(lg[mk],y[mk])
        else:
            x,y,inter=task.batch(bs,dev); lg,_=m(x,oracle_inter=inter)
            loss=F.cross_entropy(lg[:,-1,:ns],y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); sch.step()
    return m

@torch.no_grad()
def evaluate(m,task,mode,n=2000,bs=500,dev=DEV):
    m.eval(); ns=task.n_states; c=0
    for _ in range(n//bs):
        if mode=="cot":
            s,g=task.sample(bs); ans=task.trace(s,g)[:,-1]
            pre=np.concatenate([np.full((bs,1),task.BOS),s[:,None],g+task.POFF,
                                np.full((bs,1),task.SEP)],1)
            cur=torch.from_numpy(pre).long().to(dev)
            for _ in range(task.k):
                lg,_=m(cur); cur=torch.cat([cur,lg[:,-1,:ns].argmax(-1,keepdim=True)],1)
            c+=int((cur[:,-1].cpu().numpy()==ans).sum())
        else:
            x,y,inter=task.batch(bs,dev); lg,_=m(x,oracle_inter=inter)
            c+=int((lg[:,-1,:ns].argmax(-1)==y).sum())
    m.train(); return c/(n//bs*bs)
print("[ok]")

## 6 · Main run (~5 min on T4)

In [ ]:
MODES=["baseline","sum_noroles","raw_multi","hrr","cot"]
LAB={"baseline":"nothing","sum_noroles":"1 slot, plain sum","raw_multi":"3 separate slots, raw",
     "hrr":"ROLE-TAGGED latents","cot":"token chain-of-thought"}
res=[]
for Cls in [S5Task,AffineTask]:
    t=Cls(k=4,domain=120)
    print(f"\n{'='*66}\n{t.name}  k=4 (3 intermediates)  chance={t.chance:.4f}\n{'='*66}")
    for mode in MODES:
        t0=time.time(); m=fit(t,mode); a=evaluate(m,t,mode)
        res.append(dict(task=t.name,mode=mode,acc=a))
        print(f"  {mode:12s} {LAB[mode]:26s} acc {a:.4f}  [{time.time()-t0:.0f}s]",flush=True)
json.dump(res,open("results.json","w"),indent=1)

## 7 · Multi-seed

In [ ]:
print(f"{'task':<9}{'arm':<14}"+"".join(f"{f'seed {s}':>10}" for s in [0,1,2])+f"{'mean':>9}")
print("-"*66)
for Cls in [S5Task,AffineTask]:
    for mode in ["sum_noroles","raw_multi","hrr"]:
        accs=[evaluate(fit(Cls(k=4,domain=120,seed=s),mode,seed=s),
                       Cls(k=4,domain=120,seed=s),mode) for s in [0,1,2]]
        print(f"{Cls(k=4).name:<9}{mode:<14}"+"".join(f"{a:>10.4f}" for a in accs)+f"{np.mean(accs):>9.4f}",flush=True)

## 8 · THE DIAGNOSIS — what actually drives it

This section killed my original hypothesis. Run it.

A–C strip the classical binding story apart. D–F rule out scale. G–I find the real
mechanism. **Watch I**: the same transform on every slot is the control that must fail.

In [ ]:
t=S5Task(k=4,domain=120)
g=torch.Generator().manual_seed(7)
Q=[torch.linalg.qr(torch.randn(64,64,generator=g))[0].to(DEV) for _ in range(3)]
P=[torch.randperm(64,generator=g).to(DEV) for _ in range(3)]

def run(fn,tag,steps=2000):
    m=fit(t,"hrr",steps=steps,slot_fn=fn)
    with torch.no_grad():
        x,y,inter=t.batch(64,DEV); f=[m.val_embed(inter[:,i]) for i in range(3)]
        nr=float(m._memory_slots(f).norm(dim=-1).mean())
    print(f"  {tag:48s} {evaluate(m,t,'hrr'):.4f}   slot-norm {nr:6.3f}",flush=True)

print("classical binding story:")
run(lambda s,f:(lambda M:torch.stack([hrr_unbind(M,s.roles[i]) for i in range(len(f))],1))
    (sum(hrr_bind(x,s.roles[i]) for i,x in enumerate(f))), "A. bind -> superpose -> unbind")
run(lambda s,f: torch.stack([hrr_unbind(hrr_bind(x,s.roles[i]),s.roles[i]) for i,x in enumerate(f)],1),
    "B. round-trip per slot, NO superposition")
run(lambda s,f: torch.stack([hrr_bind(x,s.roles[i]) for i,x in enumerate(f)],1),
    "C. bind only, never unbound")
print("\nis it scale?")
run(lambda s,f: torch.stack(f,1),                        "D. raw values")
run(lambda s,f: torch.stack([x*0.25 for x in f],1),      "E. raw values x0.25")
print("\nis it ANY distinct per-slot transform?")
run(lambda s,f: torch.stack([x@Q[i] for i,x in enumerate(f)],1),
    "G. distinct random ORTHOGONAL per slot")
run(lambda s,f: torch.stack([x[...,P[i]] for i,x in enumerate(f)],1),
    "H. distinct random DIM-PERMUTATION per slot")
run(lambda s,f: torch.stack([x@Q[0] for x in f],1),
    "I. SAME transform every slot  <- MUST FAIL")

## 9 · Recurrent depth, tested properly

An earlier round concluded looping does not help. That run was short. Kohli et al.
(arXiv:2604.07822) report the effect is a grokking phenomenon needing zero-init
blocks and very long training. This tests that claim. **In my run it still failed
at 12,000 steps (0.2055).** Reporting the negative.

In [ ]:
t=S5Task(k=2,domain=120)
print(f"S5 k=2, chance {t.chance}")
for loops,zi,steps in [(1,False,3000),(4,False,3000),(4,True,3000),(4,True,12000)]:
    t0=time.time(); m=fit(t,"baseline",steps=steps,n_loops=loops,zero_init=zi)
    print(f"  loops={loops} zero_init={zi} steps={steps:<6} acc {evaluate(m,t,'baseline'):.4f}  [{time.time()-t0:.0f}s]",flush=True)

## 10 · What you can and cannot claim

**Can claim:** injecting intermediate values as role-tagged latents lets a small
transformer match chain-of-thought accuracy on 4-step composition with zero extra
tokens; any distinct per-slot invertible transform works; the same transform on
every slot does not; positional separation alone is insufficient. Replicated on two
algebras, three seeds.

**Cannot claim:** that the model computes its own intermediates (it is given them);
that classical role–filler binding is the mechanism (controls A–C rule it out); that
this transfers beyond toy scale.

**Open, and it is the hard one:** where do the intermediates come from at inference?